# RepairScope-X — Notebook 08 FINAL
## Independent Region-Interaction Confirmation — AR Control

**Fully standalone. No earlier ZIP is required.**

Notebook 07 v2 produced a valid **117/117 unique-trial matrix** and **39 intervention surfaces**. Its scientific signal was substantial:

- 4/8 failures showed non-uniform single-region attribution;
- 7 region pairs had pilot interaction `I >= 1/3`;
- those positive pairs covered all 3 datasets;
- untouched response text was preserved exactly whenever a patch parsed.

However, the global patch parse rate was **0.9231**, below the pre-set 0.95 gate, so we should not treat Notebook 07 as a clean final confirmation.

Crucially, all **7 positive candidate pairs** and both of their matching single-region conditions had **100% pilot parse rate**. This notebook therefore performs an **independent, pre-registered confirmation** of exactly those candidate interactions using **10 entirely new seeds**.

### Confirmatory design

Pilot Notebook 07 is used only to select the 7 candidate region pairs and to pre-register one best-single comparator for each pair. The new confirmatory seeds are:

`2801 ... 2810`

There are **20 unique intervention conditions** across the 7 candidate pairs:

- each pair;
- both constituent single-region interventions;
- duplicated singles are run only once.

Therefore:

`20 conditions × 10 new seeds = 200 new generations`

The primary confirmatory estimate remains:

`I(R_i,R_j) = R(R_i ∪ R_j) - max(R(R_i), R(R_j))`

The notebook additionally reports:

- bootstrap 95% CIs for interaction;
- a paired one-sided exact test against the **pilot-preselected best single**;
- Holm correction across the 7 candidate pairs;
- a 13-seed descriptive table combining the old 3 pilot seeds with the 10 new seeds.

### Parsing protocol v3

The generation format still uses PATCH tags, but the parser is now predefined to tolerate a missing closing tag while requiring the opening marker(s). This addresses formatting brittleness without silently changing verifier outcomes. Parser status and every generated patch are still logged.

This remains an **AR control**. The purpose is to finish the AR-side mechanism evidence cleanly before moving to DLM-native selective remasking.

Embedded payload SHA-256: `6ca1da9f9b18b2a263ecd944e9dc59bbfc205fa58c524cb00c22fc620a87553b`

In [ ]:
!pip -q install -U "transformers>=4.48,<5" "accelerate>=1.0" pandas scipy

## 1. Runtime preflight

In [ ]:
import os, gc, json, math, re, time, random, hashlib, platform, difflib
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import torch
import transformers
from scipy.stats import binomtest

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

BASE_SEED = 2800
random.seed(BASE_SEED)
np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(BASE_SEED)
    torch.cuda.empty_cache()

gc.collect()

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "Notebook 08 requires a GPU runtime."
)

GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30

print("GPU:", GPU_NAME)
print("VRAM:", round(GPU_GB, 2), "GB")
print("Allocated:", round(torch.cuda.memory_allocated()/2**30, 3), "GB")

if torch.cuda.memory_allocated()/2**30 > 1.0:
    raise RuntimeError(
        "GPU memory is already occupied. Restart the runtime and run from the top."
    )

print("Runtime preflight: PASS")

## 2. Recover and validate the embedded pilot payload

In [ ]:
import base64, zlib

EMBEDDED_PAYLOAD_SHA256 = "6ca1da9f9b18b2a263ecd944e9dc59bbfc205fa58c524cb00c22fc620a87553b"

_raw = zlib.decompress(
    base64.b64decode('''eNrtXQtz20aS/iso1VatHUv0vB+qda7iXG43V1u5rO3drasoxQXBocQSCSgkKFnn1X+/ngFfIAcgKIMkqNBKaIuYacyjv6+75/nlrBf2B5ORGZ9d/vLl7G6UdAZm2O53zy7PomQSp93kIb6MwkG/MwrTfhJfYoX4JeFn52fdMA3HJl1O2e6BpBt4dheO0r5Nb58uctsno2R4ZzP9fWyC9MYE8WTYMaNxgPV5QOD/MO4GhAXmcxilg8cgiSNIlwSQrzuJsixpOLo2aYBZK/hg0skoDsLgHt7SDcJRP70ZmrQfgYA7KMwY3upEGvj/AR4G//qvH3/67q/t7376+M8fPlwGf8re/+2/WlC4TPDZJUhG52fJJL2b2KK+Ijq4gAK+Dr6xRXsbvILPCyjt66s4L+0CMxBzb0b9Xj/KqnwJzRqOxwaatBcOxuYcGiQ118noESQ/jJL4ug1FgBwR5Byn8HfahpJ34W+X/SIrzJ0ZDftQnXvje7pc8KfzM2i6STho95LRQzjqtqEHBtDB0Lpn/Riq1E6TWxPDNxjR87NrExvoHtOdf20TPkCe9thESdy1CVuMIUUF1hhLSYUitkThbft+FA7b152zS96SQjGCCNaMYKrk+dkw6ZpBpkt/ezDxW/tBWvyCvr/4MYaaTqL0bJZqZO7740xfwlAZSTiVWtNOqLURutsLw17ItekizDtYcsS6GPKOJnF73L+OQ1ACA1kV60hMMOsxZoyhHd3VSPOox6Ie78gu6UQyVFGPdAzuEk4QQgIR2dUYdUKBIuZ6AIRBjc+SW/jNjEbJ6OwyngwGT+fV8cFQ/fiw2JjCQ1VCB9kZOogHHQ4cCrCB0SooCKobEwSVQCJ7uFzYYkTQdURgLyLoOiKoUlhSrhAWWkgpvYjAWiKuBcWUqd8zInZgMAjYCza1F5UQwXdmL/iqvVDWPDBnL1bRULuBKDMPOePAy4wD9hgH4oMCXjMOqKWx1JhirZjESGjqhQJFCgsilCZIv3goXI+H6vYyHYX9+FJolld/97Ca0v93Mr4JOpPHcUCDuzC6HQdJL7BKEF8H0Y0xY9MKgh/C6Ca46xtQ99WnQZSMwRagIDJxOp6nBUnBTQgYQvnk46AfW8gMIeVfkodgGMaPQTcZDEIAXBcAA2i6Cx//Y0n3RU71F+Lti90Lvpm/PngHv8+KchV/7McZQEcGgGgg2Sxd3yI0e+154JrgLoSX28xvXTIQNCtWa83c1G5tyozNMsBECb44rYgvvmZqaAszpJmSkjCKGDmha3wfDu+m6IpuwsEFxjiHMJegGsL+N5kAFEC5RNAJr8cZ6ZvfoAunlsZiKkqS274B/PzYCx6nGTAPIrAU8LUzGJzMUjkMJSkImENo9qSbwIexGIF3gZR7swylvF/1V5P+EXKYOEmXDd9SceyL5sIAzVdXr4LP8Pm6FfwZFNUiOUyXADat4nlmFF0JPTIzOURMJf3TBLcx1KMHDeYyTnsisN1lhoDX+WsKZd4NJmN/gmkDuvYe2zZ8Y9v1XSBkK/hkCw4wMufBg7FJgwcw11lb2AzOu7iK4efqF1feeV4o+a/2wcdkcG/JDYQsGieXxaW+sPmyLPOvoSjzb+zvV1db0YrjCiCWYTi6NaM2VLNdgWmsshdSDc9xTZlbi4laJRvNvbbcJsyTDXAYE1hx8GoFJUh5yAZwTinnSnAmFf/9kQ24888lm+96KSi+ge9BK0OwppDlMRgaAAVo9SPouYVTEoxNpubvJ9EtJL0Jh2ABB2Fkpqg4Dz4A7s0gGICE2HQzCDIe2NZNEzDWUTh0frBXCnjJUyhDdySAzW746LhNQSx3Z5K7gckkOck+AZYl+1MhIXgYilsZ4wXjzQtSLieZklQ371RIlfeoPyVBrw8kC18EN4WvWBL/c178+M5EFrn2PY5NYmOLBA096UDPRukKwc5FL7Heoqkc3/vobJ4tAaZwKWbtMpdTOUsrI7CMrEGdosnAqRMQ9GVGS1dXqfmcfvnHXELWH8t1fXLM5ZJ9ci+evw46b/qiJ+C/AlHLlV5KtiLK1S2xZmYqb8q+zgZBWS+CwndD8UDlbJKyV0Mqdmj2zUVSTj23oV8gq8r8qzWnCnNNJfh6EOF6CJjAfxRkCvAHX95A26/wPnMNZWx3DaDeUWrN49NT+U7OB5SVZgRJoY8h8AfrKqCzQdVLB3/PthkA2a5IeFEkIqdlYmxepvVB51pHJze1DpYrrbM0+LfDkiw1ClbTotCiRiHbleRrFQbTpSZZGv3ZXSGWW4PN1FYXtIZHQSqNUGyqtg1Gpy+sHvnXUZal2jM66wO8UIaaxhbqKCpZ6ijM5/qCihT37Pnh7qbuohjP33oKL78qvKyrl5YUmeKZJlO+pMnbxbB1lWtJayHWm5VLqnm5SkPl5wdRmy3zguNOUUEuKqir1XPOx5zpKcl1/XEEH3U1CVk2NzOUcrRg03piHOv6Os6C4kD4EfZHu/F7wxm4pr93Zt1+1x8kadvGTgPTDtvjSRSZsZviQSsPO96HtsiLB3jxoB+nxiLKRR5L33fMeCYyK9Mu3Me6q0tX/mxXe9QS+T+yoDHwLpzHuttipS5i27YoasvNbbGNJ+atNald4Yubo1wDRHU4bF1rvIdaF3VibX29rQ9zpDUnX2+v9lbzejUdO9uXZWsPwzs7vldu6C5/+cUC5NfzXzJkgwj3b/jLWtFSq/HsvJteu4bOteRknjz7a/EVcRK8mu6kVM9iVaQsy9O8w83ARG5ActQPB7saY7Nys0aYdfCteZy798YO4YKfid0Eyti0k9szqMzEnJ+BxGQS3djygZdlh2PHZnRv02fP59o4HQHeMEBc0+LKUfgAYEijm/Z8euJPP3/36fu/tL/7FusLot8Q9qe3s29cQ1jsQC3m6WfJ/MtHHYLAe3VN1ZtiqQ09O3LA4xrlfuz0kmuYcM42miguMZt/Fo5Uy+VR6axS87Fpub7kDSOtBWGCQiCokfCtRJAaK0qVlFy4JW+rI8cRlyIyPGTdLjZCIh4hHYUypEQDOxAcqa4yEiiy140oJTxUIaJdpnVItFbUqoxT1gr6+e8P6N9Wsepaw1a3xpOXpPHvMAvyyoyrosCXdUfIYCvAKEQGKUEG8SBDgqpiJSTRjGLMjwMZpKnIoC8BGSsqXc0y+DMdHA2kBA1kHQ1IEqkooXZ9NCF27ckRoIEeCg14HQ344J5R6WQ41ltioXgS1Y6q6w2gKNt/s0HyZphQJejyD1/HSR4lEhfBhNJimFDPDgIhmLaGQ2lJsPAttVIAXPgDBoUrqQ+MEnxQb2oTSsgJJS8UJYQwjjC2Os2ROg6YkKbChDYPJqxWmLCdwYQ1GyYMPCpGiNBgTpg4CpTQwwUg516ooDf1uV7Zr0VgmSZex0ppFPKM8NzpL6jvcuhxFU8TvP+2KGZ/XxSzL+StRiVVAKLt4sXFD/2KoMQu/SoECPMARCoFAIG/kESowN0CbdaUY7A3TPBDRyVvDuxyVUUJeQEoeRM47q8NJXN5x4YSrhHnhFIwFNINGhwHSEjTQUJPpuSFmBLSsjstwdUTDCITgTg/GpTQvZ1wcQyzf9sfI1GGj2wlPKiz2hCBLBJ6T8rYNKy7rudC4dxPXdN9CBRNUqkwsTtRqC+mkEoTLBDDHOIPdkAtZ6ju6b66VZy8JBUP3q1O2ZHKWu/JWxMQ8hbBztMVzGfwkvkM7jn8SHAge0E4huiBCXkUQCBNBQI9AeFYgUCIlEgwSigjnHgPemkeEA7m9BzhxF5LbIWDte1/oNJYzP/VEpUGY3Nnh20UuRkQDLRKKUJBQxnGUrDnA4SWAIR6ACK0JgxTraVwjr935lvBcw34odgi6KAAwQd1mRo5p+cwsHt8rIW8NYPFI78W5KysKimev9AlyPEcMAlmQyJQX8wZw6pgzUjTkEOaipxDjDaVnZOFyZYellez19yjjc5WVcx4JNeDFs6Wf0R9kxlEMiIQJ0hTjo7Dzhxw9Km50321osY3BlU8Qks2jNAWD1RVQYdeQYdn3Co3V27hUwCOkuWH1LP8kHNqT0jhDNuzrBj1Ds9CwaSQ8FZu3YzDRilvDuyHNXmib0f4CN7Vi5Dg3W4wsr6Kt57lIlJQTCQlQgtOKTkWiJCmQ4SeIPIyIEJamEm7PFdrRART+mjMCN3Xod3HMMe37enYpdD4hrALsnF6L0vlO/d7UzzBFJFcCYKEAlr2jOsW7+QTJVN7wrOTjxIbOSCk7cHCjR7HrX0f30vYxlevWr9yGvv6XXCxefYil3gnSi5w7g8pnLxAJZMXyDOdjSTljHDNwEgI7wLAxik9aajS0xej9FX1fSeqzlb+1EPvqKVAhZFgkmpCZMEGvKap+qHcll3O0j3To+dlms63VfXcgZT2nKlVVeab90zkLjSpILDCrDWWdqZMaooRZWxte6o7MqjAkS8hfop8yzcQaKTklCBGGfcu39DcjhhxAraB0MOiAR/S2zm+bXZfDYevVP6vV3U7HZ37o+raiI0xRYooxSUjWEl+DJpPGqr59GQHjtwOcImF1lQhJe0ZNuoY0HC4wZzGTpnViwo7XPOGVJkv45t2NExFPQcFNjxYAgLbYuiH4hIUYA8KiJKCSokR4IH4d/hQoewGBooFwwcODd4c1h9q8NRY3TiAQPcteZMLib8GC3Nxu8FD1dB5+33UEl4Keq+RkFTRI8EHaTg+6AkfLwMf2nppXBJABxFIsiPBB63/+s6j2OK23d2VZQDIriJNJtc36fwy0m/WbxB96+4AeReI1Qe5GxfcvWH9rk2VuyVEzG8JCbIrPMTi4o6x+2V2SYeYXdKxJGx+b8eGOOaYqrLTe1I3HWdCKJJCzz6xZ33jWgo/y8iS9Y1ybX0jawFzcEYwU1yjgjN/iGBCcAFcQ+3l27slmVVCeNac5C5YhbwEVpnhZ6a0qLUdlMvyNxs/oMU5CNW2UUsja4EZthBx8zvNBBA5OIDoSwKQU+vz4OGmH91YK2N+m/Tvw8H0VlE/rIKg0ItFiz0p7yvAr/rbmw1KmcckLQIlKxmNZ8SzoAwrgghjUmlM3B1PzQTl7n3lRu6MrBGUq5t4Nxix6nfnre8PfgYCOBOYUzX/9ASP+QRc1LYwAXMJkStRSiAmvUdKKEztfaFSIyww3j8E8H4cu0ZOtNYIgeo6vbIJntQIlucXYiewEjlQ0doW8wuuMaZK0eKDhJsAKnJwUNGTXXmpdkUgjLFGDDNGlGwqBHbvWpF1CJBmHzohdmRYzoNxMh/es2nhl28Wj3NDdM8ISLwjeK0aEVlbiNTsFtvMKxiv2c19jUMClYBphXhNU0m4111lmGNu95pKd73ZvmmF7Mdd3UQr5EQrJ0I5dFvtl0pKnHS55qTTlpIIaSalfS0XjDeVS8jBuaR5dxTsLvatCZNHzRK1tcJe8c9LXAnuOTRYUoI0l1hRZa9PbCr897FQ4tzLAehNfZHKM5cMiZJQfUt/YlWlBQru+gZKZ1U6dZfFv8qtPcieujUHr/OmLoNNbkWBd/1BuftQOMuzaY6nITXZ2ZxRlXPRcqShmOcUArWWosBtKBnZkMITgWCQxpnGzO7XJg2kDYvbfS2GqMYc5KiZw7M6yfnRAI+bsDv1oLNlSVMcJr38AqXi9UlgAWvnh72Wt+ksUHE9hyhxHgRbDx4AiXZFx2xdMmswDZDG0AA99jEJL7KMdasf+unNwtTO4DZPuoq4mYft8rqvpj45ngfqq8sYc0b6+ySGlkgtYtPEs1pxCWF/EFU45Q/PIpXmV31n9PSHmvgpPwVZuCpcl6w302vrzXgLC3vYJMQ3nHIiVYPpaR+LW/z0hN+8sJmYijMFXjrID+VlmTwU8v7rxzMqz2dsKNJm/ImV2U3PJj6eW1tWeOw4Kzk8mel198BeSMS1UhTwgJn/QiJ7JbdW3O0Y0fwA8597CxOqAvAFhAlOXbeE3nbe/TZQq1KkZ/jdm3G1koDVcyg5aSGBCAO1Y4xIuyO8ybgijcEVPRk2nxVx/36bOXhNtHLr5asDmpWX/PCS1dTcs5pacAxGVGmmFCHNhuZ2Puf4PhzeTeVAQw8uMMY5eLoEL+G6GZ6bV6twNq34DIrJyVVs/960UPSvJv3jOOiaOEldQBbEk2HHjOzoU5Qkt/1sANvFhZ3wOgjHwdXVq+AzfL5uBX+GssaQK0yXYjkibEqI/qy0bOTbIzOTY8vqJP3TBLdx8hD0RsnQZZx2dmA1wgzd7ojpawpl3g0mY3+CMO7aBOY3QNQYmsZeVM4t5GVuUO3B2KTBw6g/bQubwW2UvYrh5+oXV955Xij5r/bBSoMvUtovp4ngG/v71VWF2Tek7FVqi09PgCoLD7ogJQddEM9BF1jDSxCiRCFBtfA5xZpgojkWAgExkZ0zhA/Zz1pwviuKIC+GIk7csD9uOCQpbLtQWFMmwJUH3wAA71/R1xxSII0gBfpCSIFNzRhuCXqih/3Qw3Kb74MoFFGYFHsQJbNuxHPLEmWYgP9gL8pQiMpGc8V+Yozj212wLVdYnXy1rKWvIbg/scWO2eJjMri3U2wgZNE4uSwu9YXNNyWOfCdtpg8kVvjDd1sbLWOPLc+bp0pz4AeJJC64YkdTAs8Y5vaaOH0Y8iD7iz6Obw/BiTxO5HEg8pCYIUyR1oB7e4hac8mDNII86Gno4kQYuyKMWdfUQxRU1naPE+H2nlcGKEZUUdpkmtjXJMixLr7Zki0+Tjqp1T6row6DnSS9Ccb9LsAmTQCuySCcggIUdpgtm6tmHeOP/eHdoN97dLlHfbuwzgpeyTSP4eP/7N/D4+USdB4tq0A5oBj3pgBoWeBvMfSFk6cvRDzNBM6eujd4p1m/T+JrAMlk4Jp/tjbQEt4NMB6ZUcymKdYTmX49mf5uFPEqrqp1FQazhAZ3cfHpMRMClVoKXOJRuoerazQFs4eZCcoQ8Djy2goi7C0hmBOu5IFsxfOWie3aXJCTuSgOpvI49NL1x3X0WWh2LV6tJ7aK2EuQ8u0GcPpsxInmj5Dm14h819pSCz1LkY34FyygL3HktfAsoCcEWR+dlqxVaA45k0aR8wHuFqh8ptfXMTPAehmh2wOtTkr2+ENrh4SdePiQPFyTujyDgKvoRh2sq3iZTyxLdi1Jvr63WgDbYia55FQqRhpOul85gKIFfZmrSHM3uUA/bmRcRTS4HwxUVRG+YbT1UxL0+nE3gC+CG6CMYRg/Bvf9cT9NRuPsH6YbvJ9Et4CLm3AY/BwOwsgEyZSpxncmspUNuuGjw3wMDeni0RmO8zQ4F73ETea+nwDZgIQAuMVLOvNsCTSuS6G4zbAkp3KWVjZEmlEqICuahrvAJYDzXONdxRfBJyd3Lg34e/Zqm0TbJP+Yvyper1KlKzQpoZQDqCjDFEsqfHtAytyxbadfMOCXMcw5R4Ri7Q+WiWaI2GsqMKN7pwaAcz1LR2viBXLihRMv/A54AcI0pTBDSmjMmkwLpBG0QE+0cKKFl08LWEtOhD2niXBesFK0IbywnzDi+BaKMr4tMbiQ137Y8n5Z1dplKD9lym2RwOFjmmMFCg56iR3acHh4spJ/3bh+7MQ4K4xjhyEqdom/HxaU9LToqzJOeqrcpbORETeUdbkP/akw6EK5JAp4CfwZLLHYbv/NtncgEY7twjYsgSyJ8DOlPcwCE04kE1wdhinJ/gKr41sVu3+mfJWWqvrrE1eeuLIODWoaWwoG7qIGgeDOCvBmm8yWpBFs2bx40/HjdnSZnziZK/O7ADKfKO7lUlxZv++amkjJggXiWXmsJGIICSGxRpxj1WRq2tfM2e6XHj9zucKUg+pjqCJOKjxmbCnV+xNzbTN0Nm3jVXaoRAmMa6SwooISSRn3TKgzTXAJK9gArfDANLnOChpcFZCIKNbc3sbmYwUMUEb2ZgXuDjI+yEDYm31GeA0+j/BEDCdieB4xqBJiUOuRDHdHKHKhOJeS+ifUG0QMpFHEQF/AEJDT/fZM49vDfjwZt2fAamczOzPF9tLGPCu00wzS03wbZZ8YZjuG2aazruKv6Zk6qIojWXbqCys59YV5L2riEqiIUaYY07LhTFUe2wC12LW2nSS5RbJ9DS1gwd8L+wMo6bgdxuHg8f8sD6jpK93mIWn5pf/bxLRnxZh9PZ6MeqD57VHyMD0yN2vPjJ5GTr7vCi1All0umiMr138uT/vaBsCZEOtFIlvqGIoAPTlshymUojPJtAII1XYbvDcBnbK05NRnqjQW2OP2tWnjNgVamlPwuG2XvbZ9eaAW52fDEF7xGepptTB9bOf9pkXlrGjU1nzOm7N65apkPoPofJ0yQWHaHphwnLZpe6l+0zqtJCGL0rpazRL8+NOnHz7844efPv34P9bqf/z04e/ff/r7hx/aP3/44SN8PS3aU67jgdD6PTNObedHI+OAMEkj0BiCiLjA6AKTT1hcUnRJgcoZY5y8QegSWVXcgXaD9UyTKAFmMKOx64SpabxYJoOLe2sFDXBZt+uQ+zhIwm57fBMSLiCLwYxFkeh0uqQTStVTqoNFaDq9HpTLGKALzoSRqoujKOSh4UYS3gHMKt0hIY5A+DDpminM/vZg4rf2A5yVC/r+4scYjOIkSuepLGtNCxuGysqiUmvaCbU2Qnd7YdgLuTZdhOElkiPWtfY9w4c15dbmu/DfufoWuoDO1AzvLDO5pkUtUNk0uWvfORRZvfzcjs3D0hZwdL7kejgvIkrAF+gmD0AO4aDfGWVryLFC/JLYY51LnjNU/txm9xwUvfyVpG4vRy4V5sqSkn/Xh9dX+jXvGn3ZUKn5JTBu9VF5BSunXRW7VvGlxzO3ba0lNqRxTeNJ422rKuls462le3p6+n/xWd64''')
)

assert hashlib.sha256(_raw).hexdigest() == EMBEDDED_PAYLOAD_SHA256

UPSTREAM = json.loads(_raw.decode("utf-8"))

FAILURES = UPSTREAM["failures"]
CANDIDATE_PAIRS = UPSTREAM["candidate_pairs"]
ACTION_MAP = {
    pid: [tuple(x) for x in acts]
    for pid, acts in UPSTREAM["action_map"].items()
}
PILOT_TRIALS = UPSTREAM["pilot_selected_trials"]
UPSTREAM_GATE = UPSTREAM["notebook07_gate"]

assert len(CANDIDATE_PAIRS) == 7
assert sum(len(v) for v in ACTION_MAP.values()) == 20
assert len(PILOT_TRIALS) == 60
assert UPSTREAM_GATE["matrix_identity_passed"] is True
assert UPSTREAM_GATE["trials"] == 117
assert UPSTREAM_GATE["unique_trial_ids"] == 117
assert UPSTREAM_GATE["surface_rows"] == 39

FAILURE_MAP = {
    x["problem_id"]: x
    for x in FAILURES
}

print("Payload SHA-256:", EMBEDDED_PAYLOAD_SHA256)
print("Confirmatory failures:", len(FAILURES))
print("Candidate pairs:", len(CANDIDATE_PAIRS))
print("Unique conditions:", sum(len(v) for v in ACTION_MAP.values()))
print("Embedded pilot rows:", len(PILOT_TRIALS))
print("Pilot datasets:", sorted({x["dataset"] for x in CANDIDATE_PAIRS}))
print("Payload validation: PASS")

## 3. Frozen region definitions

In [ ]:
REGIONS = {}

for row in UPSTREAM["region_definitions"]:
    pid = row["problem_id"]
    REGIONS.setdefault(pid, []).append({
        "region_id": row["region_id"],
        "start": int(row["start"]),
        "end": int(row["end"]),
        "text": row["text"],
    })

for pid, spans in REGIONS.items():
    spans.sort(key=lambda x: x["start"])

    ids = [x["region_id"] for x in spans]
    assert len(ids) == len(set(ids))

    original = FAILURE_MAP[pid]["output"]

    for s in spans:
        assert original[s["start"]:s["end"]] == s["text"]

print("Frozen region definitions: PASS")

## 4. Strict verifier

In [ ]:
FINAL_RE = re.compile(
    r"(?im)^[ \t]*FINAL_ANSWER[ \t]*:[ \t]*"
    r"([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)"
    r"[ \t]*$"
)

NUM_RE = re.compile(
    r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?"
)

def strict_final(text):
    vals = FINAL_RE.findall(str(text))
    if not vals:
        return None

    try:
        x = float(vals[-1])
        return x if math.isfinite(x) else None
    except Exception:
        return None


def permissive_last_number(text):
    vals = NUM_RE.findall(
        str(text).replace(",", "")
    )

    if not vals:
        return None

    try:
        x = float(vals[-1])
        return x if math.isfinite(x) else None
    except Exception:
        return None


def verify_text(text, target):
    strict = strict_final(text)
    loose = permissive_last_number(text)
    target = float(target)

    passed = (
        strict is not None
        and math.isclose(
            strict,
            target,
            abs_tol=1e-8,
            rel_tol=1e-8,
        )
    )

    return {
        "passed": bool(passed),
        "strict_prediction": strict,
        "permissive_prediction": loose,
        "target": target,
    }


assert verify_text("x\nFINAL_ANSWER: 4", 4)["passed"]
assert not verify_text("x\nFINAL_ANSWER: 5", 4)["passed"]

print("Verifier tests: PASS")

## 5. Pre-registered candidate-pair table

The candidate list is frozen from Notebook 07. No new-seed outcome is used to select these pairs.

In [ ]:
candidate_df = pd.DataFrame(CANDIDATE_PAIRS)

display(
    candidate_df[
        [
            "problem_id",
            "dataset",
            "region_a",
            "region_b",
            "pilot_single_a_success",
            "pilot_single_b_success",
            "pilot_pair_success",
            "pilot_interaction",
            "pilot_best_single",
        ]
    ]
)

assert len(candidate_df) == 7
assert set(candidate_df["dataset"]) == {
    "countdown_fresh",
    "gsm8k",
    "svamp",
}

print("Candidate pre-registration: PASS")

## 6. Independent confirmatory seeds and matrix identity

In [ ]:
NEW_SEEDS = [
    2801, 2802, 2803, 2804, 2805,
    2806, 2807, 2808, 2809, 2810,
]

TEMPERATURE = 0.7
TOP_P = 0.9
MAX_NEW_TOKENS = 224

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
MODEL_REVISION = "aa8e72537993ba99e69dfaafa59ed015b17504d1"

def action_key(region_ids):
    return "+".join(region_ids)

def trial_id(pid, region_ids, seed):
    return (
        pid
        + "|"
        + action_key(region_ids)
        + "|"
        + str(seed)
    )

EXPECTED_IDS = {
    trial_id(pid, region_ids, seed)
    for pid, actions in ACTION_MAP.items()
    for region_ids in actions
    for seed in NEW_SEEDS
}

assert len(EXPECTED_IDS) == 200

print("Independent seeds:", NEW_SEEDS)
print("Unique conditions:", sum(len(v) for v in ACTION_MAP.values()))
print("Expected new trials:", len(EXPECTED_IDS))
print("Confirmatory matrix identity: PASS")

## 7. Clean output directory and run signature

In [ ]:
SAVE_TO_DRIVE = True
CLEAR_OLD_RESULTS = False

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path(
        "/content/drive/MyDrive/RepairScopeX/Notebook08_Independent_Region_Confirmation"
    )
else:
    OUT_DIR = Path(
        "/content/RepairScopeX_Notebook08_Independent_Region_Confirmation"
    )

OUT_DIR.mkdir(parents=True, exist_ok=True)

signature_payload = {
    "protocol_version": "region-confirmation-v1",
    "embedded_payload_sha256": EMBEDDED_PAYLOAD_SHA256,
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "new_seeds": NEW_SEEDS,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "max_new_tokens": MAX_NEW_TOKENS,
    "candidate_pairs": [
        {
            "problem_id": x["problem_id"],
            "region_a": x["region_a"],
            "region_b": x["region_b"],
            "pilot_best_single": x["pilot_best_single"],
        }
        for x in CANDIDATE_PAIRS
    ],
}

RUN_SIGNATURE = hashlib.sha256(
    json.dumps(
        signature_payload,
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()

TRIALS_PATH = OUT_DIR / "independent_region_trials.jsonl"
MANIFEST_PATH = OUT_DIR / "run_manifest.json"

if CLEAR_OLD_RESULTS:
    for p in [TRIALS_PATH, MANIFEST_PATH]:
        if p.exists():
            p.unlink()

if MANIFEST_PATH.exists():
    old = json.loads(
        MANIFEST_PATH.read_text(encoding="utf-8")
    )

    if old.get("run_signature") != RUN_SIGNATURE:
        raise RuntimeError(
            "Existing Notebook 08 outputs use a different configuration."
        )
else:
    MANIFEST_PATH.write_text(
        json.dumps(
            {
                "created_utc":
                    datetime.now(timezone.utc).isoformat(),
                "run_signature": RUN_SIGNATURE,
                **signature_payload,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

print("Run signature:", RUN_SIGNATURE)
print("Output:", OUT_DIR)

## 8. Load the exact AR control model revision

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

torch.cuda.empty_cache()
gc.collect()
torch.cuda.reset_peak_memory_stats()

t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    use_fast=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=torch.float16,
    low_cpu_mem_usage=True,
).to("cuda").eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("MODEL LOAD: PASS")
print("Class:", type(model).__name__)
print("Load seconds:", round(time.time() - t0, 2))
print(
    "Allocated GB:",
    round(torch.cuda.memory_allocated()/2**30, 3),
)

## 9. Forward-call instrumentation

In [ ]:
class ForwardCounter:
    def __init__(self, module):
        self.module = module
        self.calls = 0
        self.handle = None

    def _hook(self, module, inputs):
        self.calls += 1

    def __enter__(self):
        self.calls = 0
        self.handle = self.module.register_forward_pre_hook(
            self._hook
        )
        return self

    def __exit__(self, exc_type, exc, tb):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None

## 10. Exact intervention construction

In [ ]:
def spans_for_action(pid, region_ids):
    by_id = {
        x["region_id"]: x
        for x in REGIONS[pid]
    }

    spans = [
        by_id[rid]
        for rid in region_ids
    ]

    return sorted(
        spans,
        key=lambda x: x["start"],
    )


def make_masked_response(original, spans):
    labels = ["A", "B", "C"]

    pieces = []
    cursor = 0

    for idx, span in enumerate(spans):
        label = labels[idx]

        pieces.append(
            original[cursor:span["start"]]
        )
        pieces.append(
            f"<REGION_{label}>"
        )

        cursor = span["end"]

    pieces.append(
        original[cursor:]
    )

    masked = "".join(pieces)

    assert "{label}" not in masked
    assert "{i}" not in masked

    return masked


def apply_patches(original, spans, patches):
    repaired = original

    labels = ["A", "B", "C"]

    for label, span in reversed(
        list(zip(labels, spans))
    ):
        repaired = (
            repaired[:span["start"]]
            + patches[label]
            + repaired[span["end"]:]
        )

    return repaired


def untouched_chunks(original, spans):
    chunks = []
    cursor = 0

    for span in spans:
        chunks.append(
            original[cursor:span["start"]]
        )
        cursor = span["end"]

    chunks.append(
        original[cursor:]
    )

    return chunks


def chunks_appear_in_order(repaired, chunks):
    cursor = 0

    for chunk in chunks:
        if chunk == "":
            continue

        pos = repaired.find(
            chunk,
            cursor,
        )

        if pos < 0:
            return False

        cursor = pos + len(chunk)

    return True


print("Intervention construction helpers: READY")

## 11. PATCH parser v3

The parser requires each requested opening tag, but a missing closing tag is recoverable using the next PATCH tag or end-of-generation as the boundary.

This rule is frozen **before** any confirmatory generation.

In [ ]:
TAG_RE = re.compile(
    r"<\s*(/?)\s*PATCH_([ABC])\s*>",
    flags=re.IGNORECASE,
)


def parse_patches_v3(raw_output, n_patches):
    labels = ["A", "B", "C"][:n_patches]
    text = str(raw_output)

    starts = []

    for m in TAG_RE.finditer(text):
        is_close = bool(m.group(1))
        label = m.group(2).upper()

        if not is_close and label in labels:
            starts.append(
                (m.start(), m.end(), label)
            )

    patches = {}

    for label in labels:
        candidates = [
            x for x in starts
            if x[2] == label
        ]

        if not candidates:
            return None

        start_pos, content_start, _ = candidates[0]

        boundaries = []

        # Correct closing tag.
        close_re = re.compile(
            rf"<\s*/\s*PATCH_{label}\s*>",
            flags=re.IGNORECASE,
        )

        close_match = close_re.search(
            text,
            content_start,
        )

        if close_match is not None:
            boundaries.append(
                close_match.start()
            )

        # Any later opening PATCH tag also safely ends this patch.
        for s, e, other_label in starts:
            if s > start_pos:
                boundaries.append(s)

        content_end = (
            min(boundaries)
            if boundaries
            else len(text)
        )

        content = text[
            content_start:content_end
        ].strip()

        if not content:
            return None

        patches[label] = content

    return patches


# Standard parser tests.
assert parse_patches_v3(
    "<PATCH_A>x=4</PATCH_A>",
    1,
) == {"A": "x=4"}

assert parse_patches_v3(
    "<PATCH_A>x=4\n<PATCH_B>FINAL_ANSWER: 4",
    2,
) == {
    "A": "x=4",
    "B": "FINAL_ANSWER: 4",
}

assert parse_patches_v3(
    "<PATCH_A>x=4<PATCH_A><PATCH_B>FINAL_ANSWER: 4</PATCH_B>",
    1,
) == {
    "A": "x=4",
}

print("PATCH parser v3 tests: PASS")

## 12. Patch prompt

In [ ]:
PATCH_SYSTEM = (
    "You are a surgical quantitative-reasoning editor. "
    "Return only the requested PATCH blocks. "
    "Do not output REGION tags or commentary outside PATCH blocks."
)


def build_patch_prompt(failure, region_ids):
    pid = failure["problem_id"]
    spans = spans_for_action(
        pid,
        region_ids,
    )

    masked = make_masked_response(
        failure["output"],
        spans,
    )

    labels = [
        "A", "B", "C"
    ][:len(spans)]

    requested = [
        f"<PATCH_{label}>replacement text</PATCH_{label}>"
        for label in labels
    ]

    user = (
        "ORIGINAL PROBLEM:\n"
        + failure["prompt"]
        + "\n\nFAILED RESPONSE WITH OPEN REGIONS:\n"
        + masked
        + "\n\nRepair only the open region(s). "
          "After the patches are spliced back into the untouched response, "
          "the complete response must correctly solve the original problem. "
          "If a final-answer region is open, its replacement must contain "
          "`FINAL_ANSWER: <number>`.\n\n"
        + "RETURN ONLY:\n"
        + "\n".join(requested)
    )

    messages = [
        {
            "role": "system",
            "content": PATCH_SYSTEM,
        },
        {
            "role": "user",
            "content": user,
        },
    ]

    return messages, spans


# No accidental placeholders.
for pid, actions in ACTION_MAP.items():
    failure = FAILURE_MAP[pid]

    for region_ids in actions:
        messages, _ = build_patch_prompt(
            failure,
            region_ids,
        )
        full = "\n".join(
            m["content"]
            for m in messages
        )

        assert "{label}" not in full
        assert "{i}" not in full

print("Patch prompt audit: PASS")

## 13. Confirmatory intervention runner

In [ ]:
def reseed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@torch.inference_mode()
def run_intervention(
    failure,
    region_ids,
    seed,
):
    reseed(seed)

    messages, spans = build_patch_prompt(
        failure,
        region_ids,
    )

    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        rendered,
        return_tensors="pt",
    ).to(model.device)

    torch.cuda.reset_peak_memory_stats()

    t0 = time.time()

    with ForwardCounter(model) as fc:
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    wall = time.time() - t0

    new_ids = out[
        0,
        inputs.input_ids.shape[1]:,
    ]

    raw_patch_output = tokenizer.decode(
        new_ids,
        skip_special_tokens=True,
    ).strip()

    patches = parse_patches_v3(
        raw_patch_output,
        len(spans),
    )

    parse_ok = patches is not None

    if parse_ok:
        repaired = apply_patches(
            failure["output"],
            spans,
            patches,
        )

        untouched_ok = chunks_appear_in_order(
            repaired,
            untouched_chunks(
                failure["output"],
                spans,
            ),
        )

        if not untouched_ok:
            raise AssertionError(
                "Untouched response text was not preserved."
            )

        verification = verify_text(
            repaired,
            failure["target"],
        )

        a = tokenizer(
            failure["output"],
            add_special_tokens=False,
        ).input_ids

        b = tokenizer(
            repaired,
            add_special_tokens=False,
        ).input_ids

        matcher = difflib.SequenceMatcher(
            a=a,
            b=b,
            autojunk=False,
        )

        matched = sum(
            x.size
            for x in matcher.get_matching_blocks()
        )

        preservation = (
            matched / max(len(a), 1)
        )

    else:
        repaired = None
        untouched_ok = False

        verification = {
            "passed": False,
            "strict_prediction": None,
            "permissive_prediction": None,
            "target": float(
                failure["target"]
            ),
        }

        preservation = None

    return {
        "problem_id":
            failure["problem_id"],
        "dataset":
            failure["dataset"],
        "region_ids":
            list(region_ids),
        "action_key":
            action_key(region_ids),
        "seed":
            int(seed),
        "parse_ok":
            bool(parse_ok),
        "untouched_text_preserved":
            bool(untouched_ok),
        "success":
            bool(verification["passed"]),
        "verification":
            verification,
        "raw_patch_output":
            raw_patch_output,
        "repaired_output":
            repaired,
        "preservation":
            preservation,
        "actual_forward_calls":
            int(fc.calls),
        "generated_patch_tokens":
            int(new_ids.numel()),
        "wall_seconds":
            float(wall),
        "peak_vram_gb":
            float(
                torch.cuda.max_memory_allocated()
                / 2**30
            ),
        "run_signature":
            RUN_SIGNATURE,
    }

## 14. One-intervention smoke test

In [ ]:
first_pid = sorted(ACTION_MAP)[0]
smoke_failure = FAILURE_MAP[first_pid]
smoke_action = ACTION_MAP[first_pid][0]

smoke = run_intervention(
    smoke_failure,
    smoke_action,
    NEW_SEEDS[0],
)

print("Problem:", smoke["problem_id"])
print("Action:", smoke["action_key"])
print("Parse OK:", smoke["parse_ok"])
print("Success:", smoke["success"])
print("NFE:", smoke["actual_forward_calls"])

assert smoke["actual_forward_calls"] > 0
assert smoke["generated_patch_tokens"] > 0

print("CONFIRMATORY SMOKE TEST: PASS")

## 15. Resume-safe 200-trial confirmatory sweep

In [ ]:
def load_completed(path):
    done = {}

    if not path.exists():
        return done

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:

        for line_number, line in enumerate(
            f,
            start=1,
        ):
            if not line.strip():
                continue

            try:
                rec = json.loads(line)
            except json.JSONDecodeError as e:
                raise RuntimeError(
                    f"Malformed JSONL at line {line_number}."
                ) from e

            if rec.get(
                "run_signature"
            ) != RUN_SIGNATURE:
                raise RuntimeError(
                    "Found trial from another Notebook 08 configuration."
                )

            tid = rec["trial_id"]

            if tid in done:
                raise RuntimeError(
                    f"Duplicate trial_id found: {tid}"
                )

            done[tid] = rec

    return done


completed = load_completed(
    TRIALS_PATH
)

unexpected = (
    set(completed)
    - EXPECTED_IDS
)

if unexpected:
    raise RuntimeError(
        f"Unexpected saved trial IDs: {sorted(unexpected)[:5]}"
    )

print(
    "Already completed:",
    len(completed),
    "/",
    len(EXPECTED_IDS),
)

counter = 0

with open(
    TRIALS_PATH,
    "a",
    encoding="utf-8",
) as f:

    for pid in sorted(ACTION_MAP):
        failure = FAILURE_MAP[pid]

        for region_ids in ACTION_MAP[pid]:
            for seed in NEW_SEEDS:
                counter += 1

                tid = trial_id(
                    pid,
                    region_ids,
                    seed,
                )

                if tid in completed:
                    print(
                        f"[{counter}/{len(EXPECTED_IDS)}] SKIP {tid}"
                    )
                    continue

                try:
                    rec = run_intervention(
                        failure,
                        region_ids,
                        seed,
                    )

                    rec["trial_id"] = tid
                    rec["status"] = "ok"
                    rec["error"] = None

                except torch.cuda.OutOfMemoryError as e:
                    torch.cuda.empty_cache()
                    gc.collect()

                    rec = {
                        "trial_id": tid,
                        "problem_id": pid,
                        "dataset":
                            failure["dataset"],
                        "region_ids":
                            list(region_ids),
                        "action_key":
                            action_key(region_ids),
                        "seed": seed,
                        "status": "oom",
                        "error": str(e),
                        "run_signature":
                            RUN_SIGNATURE,
                    }

                except Exception as e:
                    rec = {
                        "trial_id": tid,
                        "problem_id": pid,
                        "dataset":
                            failure["dataset"],
                        "region_ids":
                            list(region_ids),
                        "action_key":
                            action_key(region_ids),
                        "seed": seed,
                        "status": "error",
                        "error":
                            f"{type(e).__name__}: {e}",
                        "run_signature":
                            RUN_SIGNATURE,
                    }

                f.write(
                    json.dumps(
                        rec,
                        ensure_ascii=False,
                    )
                )
                f.write("\n")
                f.flush()

                if rec["status"] == "ok":
                    print(
                        f"[{counter}/{len(EXPECTED_IDS)}] "
                        f"{pid} | {rec['action_key']} | seed={seed} | "
                        f"parse={rec['parse_ok']} | success={rec['success']}"
                    )
                else:
                    print(
                        f"[{counter}/{len(EXPECTED_IDS)}] "
                        f"{tid} | {rec['status']} | {rec['error']}"
                    )

print("Confirmatory sweep cell finished.")

## 16. Validate exactly 200 independent trials

In [ ]:
records = load_completed(
    TRIALS_PATH
)

missing = (
    EXPECTED_IDS
    - set(records)
)

extra = (
    set(records)
    - EXPECTED_IDS
)

print("Expected:", len(EXPECTED_IDS))
print("Found:", len(records))
print("Missing:", len(missing))
print("Extra:", len(extra))

if missing:
    raise RuntimeError(
        "Confirmatory sweep incomplete. "
        "Rerun the sweep cell; completed trials will be skipped."
    )

if extra:
    raise RuntimeError(
        "Unexpected trial IDs found."
    )

assert len(records) == 200

current = [
    records[tid]
    for tid in sorted(EXPECTED_IDS)
]

status_counts = Counter(
    x["status"]
    for x in current
)

print("Statuses:", dict(status_counts))

if status_counts.get("oom", 0):
    raise RuntimeError(
        "OOM records exist."
    )

if status_counts.get("error", 0):
    raise RuntimeError(
        "Unhandled runtime errors exist."
    )

CONFIRM_DF = pd.DataFrame(current)

assert len(CONFIRM_DF) == 200
assert CONFIRM_DF["trial_id"].nunique() == 200

parse_rate = float(
    CONFIRM_DF["parse_ok"].mean()
)

print(
    "Confirmatory parse rate:",
    round(parse_rate, 4),
)

valid_patch = CONFIRM_DF[
    CONFIRM_DF["parse_ok"]
]

if len(valid_patch):
    assert valid_patch[
        "untouched_text_preserved"
    ].all()

print("200-TRIAL CONFIRMATORY MATRIX: PASS")

## 17. Confirmatory condition-level surfaces

In [ ]:
surface_rows = []

for (
    pid,
    action,
), group in CONFIRM_DF.groupby(
    ["problem_id", "action_key"]
):
    assert len(group) == 10

    surface_rows.append({
        "problem_id": pid,
        "dataset":
            group["dataset"].iloc[0],
        "action_key": action,
        "n": len(group),
        "success_rate":
            float(group["success"].mean()),
        "parse_rate":
            float(group["parse_ok"].mean()),
        "mean_nfe":
            float(
                group[
                    "actual_forward_calls"
                ].mean()
            ),
        "mean_preservation":
            float(
                group["preservation"]
                .dropna()
                .mean()
            )
            if group[
                "preservation"
            ].notna().any()
            else math.nan,
    })

CONFIRM_SURFACE_DF = pd.DataFrame(
    surface_rows
)

assert len(CONFIRM_SURFACE_DF) == 20
assert CONFIRM_SURFACE_DF["n"].min() == 10
assert CONFIRM_SURFACE_DF["n"].max() == 10

display(CONFIRM_SURFACE_DF)

## 18. Bootstrap interaction confidence intervals

The bootstrap resamples the **10 confirmatory seed indices**, preserving the paired seed structure across the pair and both matching singles.

In [ ]:
def bootstrap_interaction(
    pid,
    region_a,
    region_b,
    n_boot=20000,
    rng_seed=8801,
):
    pair_key = (
        region_a + "+" + region_b
    )

    def vector(action):
        g = CONFIRM_DF[
            (CONFIRM_DF["problem_id"] == pid)
            & (
                CONFIRM_DF[
                    "action_key"
                ] == action
            )
        ].sort_values("seed")

        assert list(g["seed"]) == NEW_SEEDS

        return (
            g["success"]
            .astype(int)
            .to_numpy()
        )

    va = vector(region_a)
    vb = vector(region_b)
    vp = vector(pair_key)

    observed = (
        vp.mean()
        - max(
            va.mean(),
            vb.mean(),
        )
    )

    rng = np.random.default_rng(
        rng_seed
    )

    boots = np.empty(
        n_boot,
        dtype=float,
    )

    n = len(NEW_SEEDS)

    for i in range(n_boot):
        idx = rng.integers(
            0,
            n,
            size=n,
        )

        boots[i] = (
            vp[idx].mean()
            - max(
                va[idx].mean(),
                vb[idx].mean(),
            )
        )

    lo, hi = np.quantile(
        boots,
        [0.025, 0.975],
    )

    return (
        float(observed),
        float(lo),
        float(hi),
    )


print("Bootstrap helper: READY")

## 19. Independent pair confirmation + exact paired tests

For statistical testing, each pair is compared with the **best single chosen from the pilot**, not the best single chosen from the new data. This avoids using the confirmatory outcomes to select the comparator.

The one-sided exact test considers only discordant seed pairs:

- pair succeeds / comparator fails;
- pair fails / comparator succeeds.

Holm correction is then applied across the 7 candidate interactions.

In [ ]:
result_rows = []

for idx, cand in enumerate(
    CANDIDATE_PAIRS
):
    pid = cand["problem_id"]
    a = cand["region_a"]
    b = cand["region_b"]
    pair_key = a + "+" + b
    comparator = cand[
        "pilot_best_single"
    ]

    def action_group(action):
        return CONFIRM_DF[
            (CONFIRM_DF["problem_id"] == pid)
            & (
                CONFIRM_DF[
                    "action_key"
                ] == action
            )
        ].sort_values("seed")

    ga = action_group(a)
    gb = action_group(b)
    gp = action_group(pair_key)
    gc = action_group(comparator)

    assert len(ga) == 10
    assert len(gb) == 10
    assert len(gp) == 10
    assert len(gc) == 10

    sa = float(
        ga["success"].mean()
    )
    sb = float(
        gb["success"].mean()
    )
    sp = float(
        gp["success"].mean()
    )

    interaction, ci_lo, ci_hi = (
        bootstrap_interaction(
            pid,
            a,
            b,
            n_boot=20000,
            rng_seed=8900 + idx,
        )
    )

    pair_vec = (
        gp["success"]
        .astype(int)
        .to_numpy()
    )

    comp_vec = (
        gc["success"]
        .astype(int)
        .to_numpy()
    )

    pair_only = int(
        np.sum(
            (pair_vec == 1)
            & (comp_vec == 0)
        )
    )

    comp_only = int(
        np.sum(
            (pair_vec == 0)
            & (comp_vec == 1)
        )
    )

    discordant = (
        pair_only
        + comp_only
    )

    if discordant == 0:
        p_value = 1.0
    else:
        p_value = float(
            binomtest(
                pair_only,
                discordant,
                p=0.5,
                alternative="greater",
            ).pvalue
        )

    result_rows.append({
        "problem_id": pid,
        "dataset": cand["dataset"],
        "region_a": a,
        "region_b": b,
        "pilot_interaction":
            cand["pilot_interaction"],
        "pilot_best_single":
            comparator,
        "confirm_single_a_success":
            sa,
        "confirm_single_b_success":
            sb,
        "confirm_pair_success":
            sp,
        "confirm_interaction":
            interaction,
        "bootstrap_ci95_low":
            ci_lo,
        "bootstrap_ci95_high":
            ci_hi,
        "pair_only_discordant":
            pair_only,
        "comparator_only_discordant":
            comp_only,
        "raw_one_sided_p":
            p_value,
    })

PAIR_CONFIRM_DF = pd.DataFrame(
    result_rows
)

# Holm step-down adjustment.
order = np.argsort(
    PAIR_CONFIRM_DF[
        "raw_one_sided_p"
    ].to_numpy()
)

m = len(PAIR_CONFIRM_DF)
adjusted = np.empty(
    m,
    dtype=float,
)

running = 0.0

for rank, idx in enumerate(order):
    raw_p = float(
        PAIR_CONFIRM_DF.iloc[
            idx
        ]["raw_one_sided_p"]
    )

    adj = min(
        1.0,
        (m - rank) * raw_p,
    )

    running = max(
        running,
        adj,
    )

    adjusted[idx] = running

PAIR_CONFIRM_DF[
    "holm_p"
] = adjusted

PAIR_CONFIRM_DF[
    "interaction_ge_0_20"
] = (
    PAIR_CONFIRM_DF[
        "confirm_interaction"
    ] >= 0.20
)

PAIR_CONFIRM_DF[
    "bootstrap_lcb_gt_0"
] = (
    PAIR_CONFIRM_DF[
        "bootstrap_ci95_low"
    ] > 0
)

PAIR_CONFIRM_DF[
    "holm_significant_0_05"
] = (
    PAIR_CONFIRM_DF[
        "holm_p"
    ] < 0.05
)

display(
    PAIR_CONFIRM_DF.sort_values(
        "confirm_interaction",
        ascending=False,
    )
)

## 20. Independent replication gate

The AR region-interaction mechanism is considered **replicated** when all of the following hold:

1. exactly 200 independent confirmatory trials completed;
2. confirmatory parser success ≥ 98%;
3. at least 3 of the 7 preselected pairs retain interaction ≥ 0.20;
4. those replicated pairs cover at least 2 datasets.

A separate stronger-statistics flag reports whether at least one candidate has either:

- bootstrap interaction lower bound > 0; or
- Holm-adjusted one-sided paired `p < 0.05`.

The replication gate is the main progression criterion; the stronger flag is supplementary because only 10 independent seeds are used per condition.

In [ ]:
replicated_pairs = int(
    PAIR_CONFIRM_DF[
        "interaction_ge_0_20"
    ].sum()
)

replicated_datasets = int(
    PAIR_CONFIRM_DF.loc[
        PAIR_CONFIRM_DF[
            "interaction_ge_0_20"
        ],
        "dataset",
    ].nunique()
)

strong_pairs = int(
    (
        PAIR_CONFIRM_DF[
            "bootstrap_lcb_gt_0"
        ]
        |
        PAIR_CONFIRM_DF[
            "holm_significant_0_05"
        ]
    ).sum()
)

CONFIRM_GATE = {
    "independent_trials": int(
        len(CONFIRM_DF)
    ),
    "unique_trial_ids": int(
        CONFIRM_DF[
            "trial_id"
        ].nunique()
    ),
    "confirmatory_conditions": int(
        len(CONFIRM_SURFACE_DF)
    ),
    "candidate_pairs": int(
        len(PAIR_CONFIRM_DF)
    ),
    "confirmatory_parse_rate":
        parse_rate,
    "replicated_pairs_interaction_ge_0_20":
        replicated_pairs,
    "datasets_with_replicated_pairs":
        replicated_datasets,
    "strong_statistical_pairs":
        strong_pairs,
    "matrix_identity_passed": bool(
        len(CONFIRM_DF) == 200
        and CONFIRM_DF[
            "trial_id"
        ].nunique() == 200
        and len(
            CONFIRM_SURFACE_DF
        ) == 20
    ),
    "parse_rate_ge_0_98": bool(
        parse_rate >= 0.98
    ),
    "at_least_3_replicated_pairs": bool(
        replicated_pairs >= 3
    ),
    "replication_across_2_datasets": bool(
        replicated_datasets >= 2
    ),
    "strong_statistics_present": bool(
        strong_pairs >= 1
    ),
}

CONFIRM_GATE[
    "AR_REGION_INTERACTION_REPLICATED"
] = bool(
    CONFIRM_GATE[
        "matrix_identity_passed"
    ]
    and CONFIRM_GATE[
        "parse_rate_ge_0_98"
    ]
    and CONFIRM_GATE[
        "at_least_3_replicated_pairs"
    ]
    and CONFIRM_GATE[
        "replication_across_2_datasets"
    ]
)

print(
    json.dumps(
        CONFIRM_GATE,
        indent=2,
    )
)

## 21. Descriptive 13-seed combined table

This table combines the original 3 pilot seeds and the 10 independent confirmation seeds. It is **descriptive only** because the 3 pilot seeds were used to select candidate pairs.

In [ ]:
pilot_norm = []

for x in PILOT_TRIALS:
    pilot_norm.append({
        "problem_id":
            x["problem_id"],
        "dataset":
            x["dataset"],
        "action_key":
            x["action_key"],
        "seed":
            x["seed"],
        "success":
            bool(x["success"]),
        "parse_ok":
            bool(x["parse_ok"]),
        "source":
            "pilot",
    })

confirm_norm = []

for _, x in CONFIRM_DF.iterrows():
    confirm_norm.append({
        "problem_id":
            x["problem_id"],
        "dataset":
            x["dataset"],
        "action_key":
            x["action_key"],
        "seed":
            int(x["seed"]),
        "success":
            bool(x["success"]),
        "parse_ok":
            bool(x["parse_ok"]),
        "source":
            "independent_confirmation",
    })

COMBINED_DF = pd.DataFrame(
    pilot_norm
    + confirm_norm
)

combined_rows = []

for (
    pid,
    action,
), g in COMBINED_DF.groupby(
    ["problem_id", "action_key"]
):
    combined_rows.append({
        "problem_id": pid,
        "dataset":
            g["dataset"].iloc[0],
        "action_key":
            action,
        "n": len(g),
        "success_rate":
            float(
                g["success"].mean()
            ),
        "parse_rate":
            float(
                g["parse_ok"].mean()
            ),
    })

COMBINED_SURFACE_DF = pd.DataFrame(
    combined_rows
)

assert COMBINED_SURFACE_DF["n"].min() == 13
assert COMBINED_SURFACE_DF["n"].max() == 13

display(COMBINED_SURFACE_DF)

## 22. Save all Notebook 08 artifacts

In [ ]:
candidate_path = (
    OUT_DIR / "candidate_pairs_preregistered.csv"
)
surface_path = (
    OUT_DIR / "confirmatory_condition_surface.csv"
)
pair_path = (
    OUT_DIR / "independent_pair_confirmation.csv"
)
combined_path = (
    OUT_DIR / "combined_13seed_descriptive.csv"
)
gate_path = (
    OUT_DIR / "REGION_CONFIRMATION_GATE.json"
)

candidate_df.to_csv(
    candidate_path,
    index=False,
)

CONFIRM_SURFACE_DF.to_csv(
    surface_path,
    index=False,
)

PAIR_CONFIRM_DF.to_csv(
    pair_path,
    index=False,
)

COMBINED_SURFACE_DF.to_csv(
    combined_path,
    index=False,
)

gate_path.write_text(
    json.dumps(
        CONFIRM_GATE,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Saved Notebook 08 outputs:",
    OUT_DIR,
)

## 23. Artifact hashes

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


artifact_paths = [
    MANIFEST_PATH,
    TRIALS_PATH,
    candidate_path,
    surface_path,
    pair_path,
    combined_path,
    gate_path,
]

artifact_manifest = {
    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
    "run_signature":
        RUN_SIGNATURE,
    "embedded_payload_sha256":
        EMBEDDED_PAYLOAD_SHA256,
    "protocol_version":
        "region-confirmation-v1",
    "files": {
        p.name: sha256_file(p)
        for p in artifact_paths
        if p.exists()
    },
}

artifact_manifest_path = (
    OUT_DIR / "artifact_manifest.json"
)

artifact_manifest_path.write_text(
    json.dumps(
        artifact_manifest,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        artifact_manifest,
        indent=2,
    )
)

## 24. ZIP and download all Notebook 08 results

In [ ]:
import shutil

DOWNLOAD_RESULTS = True

ZIP_BASE = Path(
    "/content/RepairScopeX_Notebook08_Independent_Region_Confirmation_results"
)

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=OUT_DIR,
)

print("Created:", zip_path)
print(
    "Size MB:",
    round(
        Path(
            zip_path
        ).stat().st_size
        / 1024**2,
        3,
    ),
)

if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(zip_path)

# Progression rule

If:

`AR_REGION_INTERACTION_REPLICATED = true`

then the AR-side mechanism evidence is sufficiently developed for this project stage. Do **not** spend more notebooks merely expanding Qwen repair controls.

The next experiment should move to a **DLM-native selective-remasking runtime** and reproduce:

- failure collection;
- local/global repair surfaces;
- repair energy;
- same-error recurrence;
- region-level interventions;
- matched actual NFE.

Send back:

`RepairScopeX_Notebook08_Independent_Region_Confirmation_results.zip`